# Week 5 Lab 5: Random Forest (The Titanic Powerhouse & Pipelines)

**Goal**: Predict Titanic survival using a Random Forest to fix the high variance of Decision Trees and automatically extract the most important survival factors.

> **Why this lab matters**:
> A single Decision Tree is like asking one person for advice; a **Random Forest** is like asking a diverse crowd of 100 people and taking a vote. This is an **Ensemble** method. It drastically improves accuracy and prevents overfitting.

> **Structure**:
> We follow the **6-Phase Professional Workflow**. We use a `ColumnTransformer` inside a `Pipeline` to feed data into a `RandomForestClassifier`. Finally, we evaluate its **Cross-Validation** stability and extract **Feature Importances**.

---
## Foreword
In our final lab, we return to the **Titanic Dataset** but use a much more powerful model.

1. **Phase 1: Splitting**
2. **Phase 2: Preprocessing (ColumnTransformers)**
3. **Phase 3: Assembly (Pipeline)**
4. **Phase 4: Training (Crowd Building)**
5. **Phase 5: Evaluation (Accuracy & Cross-Validation)**
6. **Phase 6: Optimization (Feature Importance)**

### 1.1 Import Dependencies & Load Data
**Concept**: We fetch the Titanic data, extracting `Pclass`, `Sex`, `Age`, and `Fare` (ticket price).


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load Titanic Dataset
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)
df = df[['Survived', 'Pclass', 'Sex', 'Age', 'Fare']].dropna()

X = df[['Pclass', 'Sex', 'Age', 'Fare']]
y = df['Survived']

---
### 1.2 Phase 1: Data Splitting
**Concept**: Secure 20% of the dataset to evaluate the crowd's final intelligence.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

---
#### Theory: Ensemble Methods & Random Forest
A Random Forest is an **Ensemble** method. It creates many Decision Trees. To ensure the trees aren't identical (which would defeat the purpose of a "crowd"), each tree only gets to look at a **random subset** of the rows and columns. This forces different trees to learn different perspectives.

| Component | Parameter | Function |
| :--- | :--- | :--- |
| `RandomForestClassifier()` | `n_estimators=100` | Creates a forest of exactly 100 independent decision trees. |
| `OrdinalEncoder()` | `['Sex']` | We still only need Ordinal Encoding because the underlying mechanism is still trees! |

### 1.3 Phase 2 & 3: Preprocessing & Assembly
**Concept**: We build the pipeline to feed our forest.
**Solution**: A ColumnTransformer handles the text, while the remaining features pass directly into the 100 trees.


In [ ]:
preprocessor = ColumnTransformer([
    ('cat', OrdinalEncoder(), ['Sex'])
], remainder='passthrough')

workflow = Pipeline([
    ('pre', preprocessor),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42))
])

---
### 1.4 Phase 4: Training
**Concept**: When we call `.fit()`, Scikit-Learn will secretly train 100 individual decision trees.
**Solution**: Call `.fit()` on the pipeline.


In [ ]:
workflow.fit(X_train, y_train)


---
### 1.5 Phase 5: Evaluation (Accuracy & Cross-Validation)
**Concept**: When we call `.predict()`, Scikit-Learn asks all 100 trees for their prediction and takes a **majority vote**.
**Solution**: We run a prediction and immediately follow up with a 5-fold cross-validation to see if it fixed the high variance from Lab 4.


In [ ]:
y_pred = workflow.predict(X_test)
print(f"Initial Testing Accuracy: {accuracy_score(y_test, y_pred):.2%}")

scores = cross_val_score(workflow, X, y, scoring='accuracy', cv=5)
print("\nAccuracy Scores across 5 folds:", np.round(scores, 3))
print(f"Average CV Accuracy: {scores.mean():.2%}")
print(f"Standard Deviation: {scores.std():.2%} (Compare this to Lab 4's variance!)")

> **Observation**: The standard deviation is now significantly lower than a single unconstrained Decision Tree! This proves the "Wisdom of the Crowd" stabilizes predictions and prevents overfitting.

---
### 1.6 Phase 6: Optimization (Feature Importance)
**Concept**: Since we have 100 trees, we can check which feature they repeatedly found most useful for splitting data. This helps us "explain" the AI to stakeholders.
**Solution**: We extract `feature_importances_` from the trained Random Forest.

In [ ]:
# Extract the model and the transformed feature names from the pipeline
rf_model = workflow.named_steps['model']
importances = rf_model.feature_importances_

feature_names = workflow.named_steps['pre'].get_feature_names_out()
feature_names = [name.split('__')[1] for name in feature_names] # Clean names

print("--- TITANIC SURVIVAL FACTOR IMPORTANCE ---")
for name, importance in zip(feature_names, importances):
    print(f"Feature: {name:7} | Importance Score: {importance:.2%}")

**Task 1**: In Phase 3, change `n_estimators=100` to `n_estimators=5`. Rerun the cross-validation and feature importances. What happens to the variance (Standard Deviation) when the crowd is too small?

<details>
<summary><strong> Click here for Solution (Try it yourself first!)</strong></summary>

If you drop the number of trees to 5, the "Wisdom of the Crowd" is lost. The standard deviation jumps back up because 5 trees aren't enough to balance out their individual biases. 

Additionally, the Feature Importance percentages shift dramatically because the sample size of trees evaluating those features is too small. 100 estimators is the standard starting point in the industry.
</details>

---
### Summary
By using 100 trees in a pipeline, our model is less likely to "hallucinate" based on noise in the data and ensures zero data leakage. You've also verified that **Sex** and **Ticket Fare/Pclass** were the primary drivers behind the survival of Titanic passengers, while relying on the power of **Ensemble Methods**.